In [1]:

import pandas as pd
from sklearn.ensemble import IsolationForest

# Dataset
urls = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    "https://www.amazon.com",
    "https://www.microsoft.com",
    "http://example.com",
    "http://login-example.com/verify/12345",
    "http://abc.com/@#$%&123456789",
    "http://very-long-suspicious-example.com/login/verify/account"
]

df = pd.DataFrame({"url": urls})

# Feature extraction
def extract_features(url):
    return [
        len(url),
        url.count("."),
        url.count("-"),
        sum(c.isdigit() for c in url),
        sum(c in "@?=&#$%" for c in url),
        url.count("/"),
        int(url.startswith("https")),
        url.count("@")
    ]

# Extract features
features = df["url"].apply(extract_features)

feature_names = [
    "url_length",
    "dots",
    "hyphens",
    "digits",
    "special_chars",
    "slashes",
    "https",
    "at_symbol"
]

X = pd.DataFrame(features.tolist(), columns=feature_names)

# Display extracted features
print("Extracted Features:")
print(X)

# Create Isolation Forest model
model = IsolationForest(
    contamination=0.25,
    random_state=42
)

# Detect anomalies
df["anomaly"] = model.fit_predict(X)

# Convert prediction results
df["status"] = df["anomaly"].map({
    1: "Normal",
    -1: "Anomalous"
})

# Display final results
print("\nURL Anomaly Detection Results:")
print(df[["url", "status"]].to_string(index=False))


Extracted Features:
   url_length  dots  hyphens  digits  special_chars  slashes  https  at_symbol
0          22     2        0       0              0        2      1          0
1          25     2        0       0              0        2      1          0
2          22     2        0       0              0        2      1          0
3          25     2        0       0              0        2      1          0
4          18     1        0       0              0        2      0          0
5          37     1        1       5              0        4      0          0
6          29     1        0       9              5        3      0          1
7          60     1        3       0              0        5      0          0

URL Anomaly Detection Results:
                                                         url    status
                                      https://www.google.com    Normal
                                   https://www.wikipedia.org    Normal
                        